In [1]:
#this notebook is for dense retrieval which encodes corpus and queries into dense vectors, retrieves via cosine similarity using GPU 

In [2]:
#imports
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import os
import pickle
import wandb

In [4]:
#gpu check
device = "cuda" if torch.cuda.is_available() else "cpu" 
print(f"Device: {device}")

Device: cuda


In [6]:
#loading the model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model loaded


In [9]:
import ir_datasets
from tqdm import tqdm

def load_corpus():
    ds = ir_datasets.load("trec-tot/2023")
    doc_ids = []
    corpus_texts = []
    print("corpus loading")
    for doc in tqdm(ds.docs_iter()):
        doc_ids.append(doc.doc_id)
        corpus_texts.append(doc.page_title + ". " +doc.text)
    return doc_ids, corpus_texts

doc_ids, corpus_texts = load_corpus()
print(f"corpus size: {len(doc_ids)}")

corpus loading


231852it [00:15, 15037.81it/s]

corpus size: 231852


In [13]:
#encoding corpus, first save it to cache for reusing
cache_path = "corpus_embedding_minilm.nyp"

if os.path.exists(cache_path):
    print("Cache found, the embeddings are loading")
    corpus_embeddings = np.load(cache_path)
else: 
    print("corpus is encoding")
    corpus_embeddings = model.encode(
        corpus_texts,
        batch_size=256,
        show_progress_bar = True,
        convert_to_numpy = True,
        normalize_embeddings = True,
        device = device
    )
    np.save(cache_path, corpus_embeddings)
    print(f"Cache kaydedildi: {cache_path}")
print(f"corpus embedding shape: {corpus_embeddings.shape}")
    

corpus is encoding


Batches:   0%|          | 0/906 [00:00<?, ?it/s]

Cache kaydedildi: corpus_embedding_minilm.nyp
corpus embedding shape: (231852, 384)


In [20]:
import math 
def load_queries_and_qrels(split="train"):
    ds = ir_datasets.load(f"trec-tot/2023/{split}")
    queries = []
    for q in ds.queries_iter():
        queries.append({"query_id": q.query_id, "text": q.text})
    qrels = {}
    for qrel in ds.qrels_iter():
        if qrel.query_id not in qrels:
            qrels[qrel.query_id] = {}
        qrels[qrel.query_id][qrel.doc_id] = qrel.relevance
    return queries, qrels

def ndcg_at_k(ranked_doc_ids, relevant_docs, k=10):
    dcg = 0.0
    for rank, doc_id in enumerate(ranked_doc_ids[:k], start=1):
        rel = relevant_docs.get(doc_id, 0)
        dcg += rel / math.log2(rank + 1)
    ideal = sorted(relevant_docs.values(), reverse=True)[:k]
    idcg = sum(rel/math.log2(i+2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(ranked_doc_ids, relevant_docs, k=100):
    total_relevant = sum(1 for v in relevant_docs.values() if v > 0)

    if total_relevant ==0:
        return 0.0
    retrieved_relevant = sum(1 for doc_id in ranked_doc_ids[:k] if relevant_docs.get(doc_id, 0) > 0)
    return retrieved_relevant / total_relevant

def evaluate(run, qrels, k_ndcg=10, k_recall = 100):
    ndcg_scores = []
    recall_scores = []
    for qid, results in run.items():
        relevant = qrels.get(qid, {})
        ranked_ids = [doc_id for doc_id, _ in results]
        ndcg_scores.append(ndcg_at_k(ranked_ids, relevant, k_ndcg))
        recall_scores.append(recall_at_k(ranked_ids, relevant, k_recall))
    return {f"nDCG@{k_ndcg}": np.mean(ndcg_scores), f"Recall@{k_recall}": np.mean(recall_scores), "_per_query_ndcg": ndcg_scores, "_per_query_recall": recall_scores,
           }

In [21]:
#loading queries and qrels

print("loading queries")
train_queries, train_qrels = load_queries_and_qrels("train")
dev_queries, dev_qrels = load_queries_and_qrels("dev")
all_queries = train_queries + dev_queries
all_qrels = {**train_qrels, **dev_qrels}
print(f"total queries: {len(all_queries)}")

#query embedding
print("encoding queries")
query_texts = [q["text"] for q in all_queries]
query_ids = [q["query_id"] for q in all_queries]
query_embeddings = model.encode(
    query_texts,
    batch_size=256,
    show_progress_bar = True,
    convert_to_numpy=True,
    device=device
)

print(f"query embeddings shape: {query_embeddings.shape}")

#cosine similarity and retrieval
print("retrieving")
sim_matrix = query_embeddings @ corpus_embeddings.T
run_dense = {}
for i, qid in enumerate (tqdm(query_ids)):
    scores = sim_matrix[i]
    top_indices = np.argsort(scores)[::-1][:100]
    run_dense[qid] = [(doc_ids[j], float(scores[j])) for j in top_indices]

loading queries
total queries: 300
encoding queries


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

query embeddings shape: (300, 384)
retrieving


100%|████████████████████████████████████████████████████████████████████████████████| 300/300 [00:03<00:00, 84.76it/s]


In [22]:
#evaluate
scores_dense = evaluate(run_dense, all_qrels)
print(f"Dense Retrieval Results")
print(f"nDCG@10:    {scores_dense['nDCG@10']:.4f}")
print(f"Recall@100: {scores_dense['Recall@100']:.4f}")

Dense Retrieval Results
nDCG@10:    0.0313
Recall@100: 0.1533


In [23]:
#logging WANDB
import os
wandb.login(key=os.environ.get("WANDB_API_KEY"))
wandb.init(project="is584-tot-retrieval", entity="ozgukan", name="dense_minilm")
wandb.log({
    "nDCG@10": scores_dense["nDCG@10"],
    "Recall@100": scores_dense["Recall@100"],
    "model": "all-MiniLM-L6-v2"
})
wandb.finish()
print("logged!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ozgu.ozkan\_netrc.
wandb: Currently logged in as: ozgukan (emredeniztaylan44-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Recall@100,▁
nDCG@10,▁
Recall@100,0.15333
model,all-MiniLM-L6-v2
nDCG@10,0.03125


logged!


In [27]:
#this cell is for wilcoxon test
from scipy.stats import wilcoxon

#load the best bm25 run per-query
import pickle
with open("bm25_run_k1_1.5_b_0.75.pkl", "rb") as f:
    run_bm25 = pickle.load(f)

#calculate bm25 per query scores
bm25_eval = evaluate(run_bm25, all_qrels)
bm25_ndcg = bm25_eval["_per_query_ndcg"]
bm25_recall = bm25_eval["_per_query_recall"]

dense_ndcg = scores_dense["_per_query_ndcg"]
dense_recall = scores_dense["_per_query_recall"]

#wilcoxon test for ndcg@10
stat_ndcg, p_ndcg = wilcoxon(bm25_ndcg, dense_ndcg)
print(f"Wilcoxon Test for nDCG@10")
print(f"BM25 mean:  {sum(bm25_ndcg)/len(bm25_ndcg):.4f}")
print(f"Dense mean: {sum(dense_ndcg)/len(dense_ndcg):.4f}")
print(f"p-value: {p_ndcg:.4f}")
print(f"Is it meaningful (p<0.05)? {'YES' if p_ndcg < 0.05 else 'NO'}")

print()

#wilcoxon test for recall@100
stat_recall, p_recall = wilcoxon(bm25_recall, dense_recall)
print(f"Wilcoxon Test for Recall@100")
print(f"BM25 mean:  {sum(bm25_recall)/len(bm25_recall):.4f}")
print(f"Dense mean: {sum(dense_recall)/len(dense_recall):.4f}")
print(f"p-value: {p_recall:.4f}")
print(f"Is it meaningful (p<0.05)? {'YES' if p_recall < 0.05 else 'NO'}")

Wilcoxon Test for nDCG@10
BM25 mean:  0.0408
Dense mean: 0.0313
p-value: 0.4672
Is it meaningful (p<0.05)? NO

Wilcoxon Test for Recall@100
BM25 mean:  0.1100
Dense mean: 0.1533
p-value: 0.0851
Is it meaningful (p<0.05)? NO


In [29]:
#wandb log
wandb.login(key=os.environ.get("WANDB_API_KEY"))
wandb.init(project="is584-tot-retrieval", entity="ozgukan", name="wilcoxon_bm25_vs_dense")
wandb.log({
    "nDCG@10_bm25": sum(bm25_ndcg)/len(bm25_ndcg),
    "nDCG@10_dense": sum(dense_ndcg)/len(dense_ndcg),
    "p_value_ndcg": p_ndcg,
    "Recall@100_bm25": sum(bm25_recall)/len(bm25_recall),
    "Recall@100_dense": sum(dense_recall)/len(dense_recall),
    "p_value_recall": p_recall,
})
wandb.finish()
print("logged!")

Recall@100_bm25,▁
Recall@100_dense,▁
nDCG@10_bm25,▁
nDCG@10_dense,▁
p_value_ndcg,▁
p_value_recall,▁
Recall@100_bm25,0.11
Recall@100_dense,0.15333
nDCG@10_bm25,0.04081
nDCG@10_dense,0.03125
p_value_ndcg,0.46715


logged!
